# DCT Laboratory — Volume II, Chapter 12
## Multi-Objective Enterprise Optimization
**Seed `26212`** · Companion to Chapter 12 and **AXIOM Lab 2.12** (module **AXIOM-12**) · Mirrored in `DCT_V2_Ch12_Lab.xlsx`

This laboratory reproduces **Worked Example 12.1, *Whose Objective Runs the Automation Program?***, with the book's
declarations, and works the five steps of AXIOM Lab 2.12: enumerate, price the frontier, bound the reserve, choose, and
add uncertainty. The program is Chapter 5's automation program; the new declarations are the three owners' objectives.
The model is deterministic: for each schedule of rounds and each reserve, the best payouts solve a small linear
program, solved here by SciPy's HiGHS as in the book, and no number below depends on a random draw. The lab's own seed,
26212, draws only Panel 6's illustrations of how frontiers are computed (random weights and an evolutionary search).
Lab step 5 reads Chapter 9's cost–chance frontier as the book prints it (Online Appendix Example 10.B.2) instead of
re-simulating its 100,000 paths. Every Meridian number below is the book's (Table 12.3, Figures 12.1–12.3, Online
Appendix Example 12.B.2); the final cell checks them. Money is in \$ million.

## The declarations (Worked Example 12.1, Step 1; Online Appendix Remark 12.B.1)

**Chapter 5's program, unchanged.** The state is year-end cash $q_k$ and efficiency $e_k$, from $q_0 = 620$ and
$e_0 = 71$. A round of automation in year $k$, $z_k \in \{0, 1\}$, costs \$120 million and halves the remaining gap to
85 points, $e_k = e_{k-1} + z_k(85 - e_{k-1})/2$; the run-rate savings are $S_k = 10(e_k - 71)$, \$10 million a year
for each point above 71; payouts $p_k \ge 0$ are free, and cash moves as $q_k = q_{k-1} - 120z_k + S_k - p_k$. The
board's commitments: cash stays above \$350 million at every year end, reaches \$470 million before a round, and ends
year 6 at \$470 million. Payouts are discounted at the 10 percent hurdle rate, $\beta = 1/1.1$; cash earns nothing.

**The three owners' objectives (new declarations).** The chief financial officer judges a plan by the present value
of its payouts, $f_1 = \sum_{k=1}^{6}\beta^{k}p_k$; the chief operating officer by the run-rate savings it leaves in
year 6, $f_2 = S_6$, which depends only on the number of rounds $n$, $S_6 = 140 - 140/2^{n}$; the treasury by the
**reserve** $b$ by which every cash commitment is kept: $q_k \ge 350 + b$ in every year, $q_{k-1} \ge 470 + b$ before a
round in year $k$, and $q_6 \ge 470 + b$. Chapter 5's value is one weighted sum of the first two, $f_1 + w f_2$ with
$w = \beta^{6}/0.10 = 5.645$: a dollar a year of year-6 savings, continued forever, priced in dollars of payouts today.
It ignores the third.

**The vector problem (Definition 12.1).** Maximize $(f_1, f_2, f_3)$ over the schedules of whole rounds, the payouts
and the reserve. For a schedule and a reserve, the best payouts solve a linear program in $p_1, \dots, p_6$ with at
most twelve constraints, solved, as in the book, by SciPy's HiGHS; its multipliers price the reserve. The cell solves
all $64 \times 151 = 9{,}664$ programs, every schedule at every reserve on a \$1 million grid from 0 to \$150 million,
and checks them against Chapter 5's closed form: pay out, every year, everything above the **war chest**, the least
year-end cash from which the rest of the schedule keeps every commitment, now with every commitment raised by $b$:
$R_6 = 470 + b$ and $R_{k-1} = \max\{R_k + 120z_k - S_k,\ (470 + b)z_k,\ 350 + b\}$, with no floor term for $R_0$;
the schedule keeps every commitment when $620 \ge R_0$.

In [ ]:
import itertools
import numpy as np
from scipy.optimize import linprog
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26212        # the lab's seed: it draws only Panel 6's random weights and evolutionary search

K, R = 6, 0.10                                          # six years; the board's hurdle rate
BETA = 1.0 / (1.0 + R)
Q0, E0, CEIL, VAL = 620.0, 71.0, 85.0, 10.0            # today's cash and efficiency; the ceiling; $10m a year a point
COST, FLOOR, PRE, Q_END = 120.0, 350.0, 470.0, 470.0   # a round; the floor; cash before a round; cash at the end
W_BOARD = BETA ** K / R                                # Chapter 5's price of a dollar a year of year-6 savings
SCHEDULES = list(itertools.product([0, 1], repeat=K))  # the 64 schedules (z_1, ..., z_6), in binary order
NR = np.array([sum(z) for z in SCHEDULES])             # rounds in each schedule
NB = 151                                               # reserves 0, 1, ..., 150 ($1m grid)

def years(z):
    # the years of a schedule's rounds as the book prints them: three or more in a row as a range
    ys = [k + 1 for k in range(K) if z[k]]
    out, i = [], 0
    while i < len(ys):
        j = i
        while j + 1 < len(ys) and ys[j + 1] == ys[j] + 1:
            j += 1
        out += [f"{ys[i]}–{ys[j]}"] if j - i >= 2 else [str(y) for y in ys[i:j + 1]]
        i = j + 1
    return ", ".join(out) or "none"

def savings(z):
    # a round halves the gap to 85 points; $10m a year for each point above 71
    e, out = E0, []
    for zk in z:
        if zk:
            e = e + (CEIL - e) / 2
        out.append(VAL * (e - E0))
    return np.array(out)

S6 = np.array([savings(z)[-1] for z in SCHEDULES])     # the COO's objective: run-rate savings in year 6

def sv(x):
    # savings as the book prints them: up to four decimals
    return f"{x:.4f}".rstrip("0").rstrip(".")

def payouts_lp(z, b=0.0):
    # The best payouts for schedule z with every commitment kept with b to spare (SciPy/HiGHS):
    # max sum_k beta^k p_k  s.t.  q_k >= 350 + b,  q_(k-1) >= 470 + b before a round in year k,  q_6 >= 470 + b,
    # where q_k is the cash with no payouts less p_1 + ... + p_k, and p >= 0.
    # Returns (present value, payouts, year-end cash, price of the reserve), or None if no payouts keep them.
    S = savings(z)
    base = Q0 + np.cumsum(S - COST * np.array(z, float))
    if z[0] and Q0 < PRE + b - 1e-9:                    # a year-1 round needs 470 + b today
        return None
    rows, rhs = [], []
    for k in range(K):
        row = np.zeros(K)
        row[:k + 1] = 1.0
        rows.append(row); rhs.append(base[k] - (FLOOR + b))             # the floor at the end of year k + 1
        if k + 1 < K and z[k + 1]:
            rows.append(row.copy()); rhs.append(base[k] - (PRE + b))    # the cash a round in year k + 2 needs
    rows.append(np.ones(K)); rhs.append(base[-1] - (Q_END + b))          # the end condition
    res = linprog(-BETA ** np.arange(1, K + 1), A_ub=np.array(rows), b_ub=np.array(rhs),
                  bounds=[(0, None)] * K, method="highs")
    if res.status != 0:
        return None
    # a dollar more of reserve lowers every right-hand side by a dollar: the payouts lose the multipliers' sum
    return float(-res.fun), res.x, base - np.cumsum(res.x), float(-np.sum(res.ineqlin.marginals))

def war_chest(z, b=0.0):
    # Chapter 5's war chest with every commitment raised by b: R_6 = 470 + b,
    # R_(k-1) = max(R_k + 120 z_k - S_k, (470 + b) z_k, 350 + b), no floor term for R_0; pay out everything above it
    S = savings(z)
    Rw = np.empty(K + 1)
    Rw[K] = Q_END + b
    for k in range(K, 0, -1):
        need = max(Rw[k] + COST * z[k - 1] - S[k - 1], (PRE + b) * z[k - 1])
        Rw[k - 1] = max(need, FLOOR + b) if k > 1 else need
    if Q0 < Rw[0] - 1e-9:
        return None
    start = np.concatenate([[Q0], Rw[1:K]])            # each later year opens with the war chest left
    p = start - COST * np.array(z, float) + S - Rw[1:]
    return float(BETA ** np.arange(1, K + 1) @ p), p, Rw

LP = {}                                                # (schedule, reserve) -> the payout program's answer
for i, z in enumerate(SCHEDULES):
    for b in range(NB):
        out = payouts_lp(z, float(b))
        if out is not None:
            LP[i, b] = out
WC_SAME, WC_GAP = True, 0.0
for i, z in enumerate(SCHEDULES):
    for b in range(NB):
        wc = war_chest(z, float(b))
        if (wc is None) != ((i, b) not in LP):
            WC_SAME = False
        elif wc is not None:
            WC_GAP = max(WC_GAP, abs(wc[0] - LP[i, b][0]), float(np.max(np.abs(wc[1] - LP[i, b][1]))))
N_PROGRAMS = len(SCHEDULES) * NB
print(f"{len(SCHEDULES)} schedules x {NB} reserves = {N_PROGRAMS:,} payout programs; {len(LP):,} keep every commitment")
print(f"the war chest against the linear programs: the same feasibility every time: {WC_SAME}; largest difference "
      f"in a payout or a value {WC_GAP:.1e}")
print(f"Chapter 5's price of savings: beta^6 / 0.10 = {W_BOARD:.4f} dollars of payouts today per dollar a year")
print("year-6 savings by number of rounds, 140 - 140 / 2^n: " + ", ".join(f"{n}: {sv(140 - 140 / 2 ** n)}" for n in range(K + 1)))

## Panel 1 — Enumerate (Theorem 12.1; Worked Example 12.1, Step 2; Figure 12.1(a): all 64 schedules keep the commitments, 59 are dominated; five are efficient, from two rounds worth \$384.17m with \$105m a year to six worth \$106.59m with \$137.81m; the ideal point is out of reach)

AXIOM Lab 2.12, step 1. With no reserve, solve the payouts for every schedule of rounds, place each schedule by its
payouts and its savings, and mark the dominated ones: a schedule is dominated when another pays at least as much and
saves at least as much, and more of one (Definition 12.1). By Theorem 12.1(i), every dominated schedule is dominated by
an efficient one, so the board can set it aside whoever proposes it. Savings depend only on the number of rounds, so
only the best schedule with each number of rounds can be efficient. Theorem 12.1(ii) is how a board already speaks:
"maximize payouts, provided savings reach \$122.5 million a year and the reserve \$50 million" bounds every objective but
one, and its answer is a point of the frontier.

In [ ]:
P0 = np.array([LP[i, 0][0] for i in range(64)])        # each schedule's best payouts, present value, no reserve
N_FEAS0 = sum((i, 0) in LP for i in range(64))

def dominated(F, tol=1e-9):
    # which rows of F (objectives, all maximized) another row dominates: as good in every one, better in one
    F = np.asarray(F, float)
    ge = np.all(F[:, None, :] >= F[None, :, :] - tol, axis=2)      # ge[j, i]: row j at least as good as row i
    gt = np.any(F[:, None, :] > F[None, :, :] + tol, axis=2)
    return np.any(ge & gt, axis=0)

def dominates(a, c, tol=1e-9):
    return bool(np.all(a >= c - tol) and np.any(a > c + tol))

F2 = np.c_[P0, S6]
DOM2 = dominated(F2)
EFF2 = sorted(np.nonzero(~DOM2)[0].tolist(), key=lambda i: NR[i])
BEST_N = {n: max((i for i in range(64) if NR[i] == n), key=lambda i: (P0[i], -i)) for n in range(K + 1)}
print(f"{N_FEAS0} of 64 schedules keep every commitment with no reserve: {DOM2.sum()} dominated, {len(EFF2)} efficient")
print("\nthe best schedule with each number of rounds:")
print("rounds  years              payouts (PV)  savings   efficient  payouts, years 1-6")
for n in range(K + 1):
    i = BEST_N[n]
    print(f"{n:6d}  {years(SCHEDULES[i]):17s}  {P0[i]:12.2f}  {S6[i]:8.4f}  {'yes' if not DOM2[i] else 'no':9s}  "
          + ", ".join(f"{x:g}" for x in np.round(LP[i, 0][1], 4)))
I3 = BEST_N[3]
print("three rounds, year-end cash: " + ", ".join(f"{x:g}" for x in np.round(LP[I3, 0][2], 4))
      + f"; {LP[I3, 0][1].sum():g} paid in all")
print(f"one round ({P0[BEST_N[1]]:.2f}, {sv(S6[BEST_N[1]])} a year) is dominated by two ({P0[BEST_N[2]]:.2f}, "
      f"{sv(S6[BEST_N[2]])} a year): on that, the CFO and the COO agree")
COVERED = all(any(dominates(F2[e], F2[j]) for e in EFF2) for j in np.nonzero(DOM2)[0])
print(f"Theorem 12.1(i): each of the {DOM2.sum()} dominated schedules is dominated by an efficient one: {COVERED}")
IDEAL2 = F2.max(axis=0)
IDEAL2_HIT = bool(np.any(np.all(F2 >= IDEAL2 - 1e-9, axis=1)))
print(f"the ideal point, payouts {IDEAL2[0]:.2f} with savings {IDEAL2[1]:.4f} a year, is attained by some schedule: "
      f"{IDEAL2_HIT} (Theorem 12.1(iii): every efficient plan gives up something)")

def best_with(bound_S, b=0):
    # Theorem 12.1(ii): the most payouts with savings of at least bound_S and a reserve of b
    ok = [i for i in range(64) if S6[i] >= bound_S - 1e-9 and (i, b) in LP]
    return max(ok, key=lambda i: (LP[i, b][0], -i))

EPS_A, EPS_B = best_with(122.5), best_with(122.5, 50)
EPS_UNIQUE = sum(abs(LP[i, 0][0] - LP[EPS_A, 0][0]) < 1e-9 for i in range(64) if S6[i] >= 122.5 - 1e-9) == 1
print(f"maximize payouts with savings of at least 122.5 a year: rounds in years {years(SCHEDULES[EPS_A])}, "
      f"{LP[EPS_A, 0][0]:.2f}, the only answer: {EPS_UNIQUE}, hence efficient")
print(f"... and a reserve of $50 million: rounds in years {years(SCHEDULES[EPS_B])}, {LP[EPS_B, 50][0]:.2f}")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
dom = np.nonzero(DOM2)[0]
ax.scatter(S6[dom], P0[dom], s=14, color="#9E9E9E", zorder=2, label=f"dominated ({len(dom)})")
ax.plot(S6[EFF2], P0[EFF2], "o-", color="#1F4E79", lw=1.8, ms=5, zorder=3, label=f"efficient ({len(EFF2)})")
for i in EFF2:
    ax.text(S6[i] + 2.0, P0[i] + 5.0, f"{NR[i]} rounds" if i == EFF2[0] else f"{NR[i]}", fontsize=8, color="#1F4E79")
ax.plot([IDEAL2[1]], [IDEAL2[0]], marker="*", ms=13, color="#E65100", ls="none", zorder=4)
ax.text(IDEAL2[1] + 3.0, IDEAL2[0], "ideal:\nout of reach", fontsize=8, color="#E65100", va="center")
ax.annotate("one round: dominated by two", xy=(S6[BEST_N[1]], P0[BEST_N[1]]), xytext=(8, 310), fontsize=8,
            arrowprops=dict(arrowstyle="-", lw=0.6, color="#333333"))
ax.set(xlim=(-5, 165), ylim=(0, 430), xlabel="run-rate savings in year 6 ($ million a year)",
       ylabel="payouts, present value ($ million)", title="Five of the 64 schedules are efficient (seed 26212)")
ax.legend(frameon=False, fontsize=8, loc="lower left"); plt.tight_layout(); plt.show()

## Panel 2 — Price the frontier (Theorem 12.2(i), (iv); Worked Example 12.1, Step 3; Figure 12.1(b): between neighbouring efficient plans a dollar a year of savings costs \$3.07, \$7.95, \$16.93 and \$36.68 of payouts; Chapter 5's price, \$5.645, picks three rounds, \$1,021.93m)

AXIOM Lab 2.12, step 2. Between two neighbouring efficient plans, the price of savings is the payouts given up per
dollar a year of savings gained. A weighted sum $f_1 + w f_2$ prices a dollar a year of savings at $w$ dollars of
payouts, and picks the plan whose neighbouring prices bracket $w$; with positive weights its answer is efficient
(Theorem 12.2(i)). Chapter 5's board value is the weighted sum with $w = \beta^{6}/0.10$, the year-6 savings continued
forever. Its answer, three rounds, is a choice of price, not a fact about the program: a board that valued savings more
would run more rounds.

In [ ]:
RATES = []
for a, c in zip(EFF2[:-1], EFF2[1:]):
    RATES.append(dict(frm=int(NR[a]), to=int(NR[c]), dP=float(P0[a] - P0[c]), dS=float(S6[c] - S6[a]),
                      rate=float((P0[a] - P0[c]) / (S6[c] - S6[a]))))
print("between neighbouring efficient plans, payouts given up for savings gained:")
for r in RATES:
    print(f"  {r['frm']} -> {r['to']} rounds: {r['dP']:6.2f} for {r['dS']:7.4f} a year = {r['rate']:6.2f} per dollar a year")
edges = [0.0] + [r["rate"] for r in RATES] + [np.inf]
print("a weighted sum, payouts + w x savings, picks")
for k, i in enumerate(EFF2):
    print(f"  {NR[i]} rounds for w from {edges[k]:.2f} to {edges[k + 1]:.2f}")
INCREASING = all(a["rate"] < c["rate"] for a, c in zip(RATES[:-1], RATES[1:]))
print(f"the prices rise along the frontier: {INCREASING}; with no reserve, every efficient plan wins some weighted sum")
BOARD_V = {int(NR[i]): float(P0[i] + W_BOARD * S6[i]) for i in EFF2}
BOARD_I = max(range(64), key=lambda i: P0[i] + W_BOARD * S6[i])
print(f"\nChapter 5's price, w = {W_BOARD:.4f}: board values " + ", ".join(f"{n} rounds {v:,.2f}" for n, v in BOARD_V.items()))
print(f"the board's weighted sum picks rounds in years {years(SCHEDULES[BOARD_I])}: {P0[BOARD_I]:.2f} of payouts + "
      f"{W_BOARD:.4f} x {S6[BOARD_I]:g} of savings = {P0[BOARD_I] + W_BOARD * S6[BOARD_I]:,.2f}")
print(f"a price below {RATES[0]['rate']:.2f} would choose two rounds, and one above {RATES[1]['rate']:.2f} four")

ws_ = np.exp(np.linspace(np.log(1.0), np.log(100.0), 801))
pick = [NR[max(range(64), key=lambda i: P0[i] + w * S6[i])] for w in ws_]
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.step(ws_, pick, where="post", color="#1F4E79", lw=2)
ax.set_xscale("log")
for r in RATES:
    ax.axvline(r["rate"], color="#999999", lw=0.8, ls=":")
    ax.text(r["rate"] * 1.06, r["frm"] + 0.5, f"{r['rate']:.2f}", fontsize=8, color="#616161", va="center")
ax.plot([W_BOARD], [3], marker="o", ms=7, mfc="white", mec="#E65100", mew=1.6, ls="none", zorder=5)
ax.annotate(f"Chapter 5's board: {W_BOARD:.3f}\nthree rounds, {BOARD_V[3]:,.2f}", xy=(W_BOARD, 3), xytext=(1.25, 4.6),
            fontsize=8, color="#E65100", arrowprops=dict(arrowstyle="-", lw=0.7, color="#E65100"))
ax.set_xticks([1, 2, 5, 10, 20, 50, 100], ["1", "2", "5", "10", "20", "50", "100"]); ax.minorticks_off()
ax.set(xlim=(1, 100), ylim=(1.5, 6.5), yticks=[2, 3, 4, 5, 6],
       xlabel="price of a dollar a year of savings ($ of payouts, log scale)", ylabel="rounds in the plan it picks",
       title="The board's price of savings, 5.645, picks three rounds (seed 26212)")
plt.tight_layout(); plt.show()

## Panel 3 — Bound the reserve (Theorem 12.2(iv); Worked Example 12.1, Step 4; Figure 12.2: \$0.909 of payouts per dollar of reserve, for every schedule and reserve; above \$85m and \$100m the rounds move later, at extra costs of \$4.02m and \$24.38m)

AXIOM Lab 2.12, step 3. Raise the reserve, read its multiplier, and find where the schedule of rounds must change. The
multiplier of the bound on $f_3$ is the sum of the payout program's multipliers, because a dollar more of reserve takes
a dollar from the slack of every commitment. The war chest shows why it is the same everywhere: every term of the
recursion rises by $b$, so the war chest rises by $b$ in every year; the payouts after year 1 do not change and the
year-1 payout falls by $b$. The reserve is a year-1 payout held back until the end, worth $\beta = \$0.909$ of payouts
per dollar, whatever the schedule. A schedule keeps the reserve only while today's cash covers its war chest,
$620 \ge R_0 + b$. After a round in year 1, the cash before a year-2 round is at most $620 - 120 + 70 = 570$, so rounds
in years 1 and 2 need $b \le 100$; after rounds in years 1 and 2, the cash before a year-3 round is at most
$570 - 120 + 105 = 555$, so rounds in years 1–3 need $b \le 85$. Beyond these, the best plan with the same number of
rounds moves a round later, and the payouts fall by more than the price: the trade-off surface of Theorem 12.2(iv)
jumps.

In [ ]:
PRICES = np.array([v[3] for v in LP.values()])
print(f"the reserve's price from the multipliers of all {len(LP):,} feasible programs: {PRICES.min():.4f} to "
      f"{PRICES.max():.4f} of payouts per dollar (beta = {BETA:.4f})")
Z3 = (1, 1, 1, 0, 0, 0)
for b in (0.0, 50.0):
    v, p, Rw = war_chest(Z3, b)
    print(f"rounds in years 1-3, reserve {b:3.0f}: war chest R_0..R_6 " + ", ".join(f"{x:g}" for x in Rw)
          + "; payouts " + ", ".join(f"{x:g}" for x in np.round(p, 4)) + f"; value {v:.2f}")
BMAX = np.array([max(b for b in range(NB) if (i, b) in LP) for i in range(64)])
print("the largest reserve a schedule keeps: " + "; ".join(f"{v}: {int(np.sum(BMAX == v))} schedules" for v in sorted(set(BMAX))))
S_ONE, S_TWO = savings((1, 0, 0, 0, 0, 0))[0], savings((1, 1, 0, 0, 0, 0))[1]
CASH2 = Q0 - COST + S_ONE
CASH3 = CASH2 - COST + S_TWO
print(f"after a year-1 round, the cash before a year-2 round is at most {Q0:.0f} - {COST:.0f} + {S_ONE:.0f} = {CASH2:.0f}: "
      f"rounds in years 1 and 2 need b <= {CASH2 - PRE:.0f}")
print(f"after rounds in years 1 and 2, the cash before a year-3 round is at most {CASH2:.0f} - {COST:.0f} + {S_TWO:.0f} = "
      f"{CASH3:.0f}: rounds in years 1-3 need b <= {CASH3 - PRE:.0f}")
print(f"no plan keeps more than {BMAX.max()}: before the first round, or at the end without one, the cash is at most "
      f"{Q0:.0f} against {PRE:.0f} plus the reserve")

def best_at(n, b):
    # the best schedule with n rounds at reserve b, the first in binary order on ties (as the book's engine)
    best = None
    for i in range(64):
        if NR[i] == n and (i, b) in LP and (best is None or LP[i, b][0] > LP[best, b][0] + 1e-12):
            best = i
    return best

ENV3 = [best_at(3, b) for b in range(NB)]
PHI = [max(LP[i, b][0] for i in range(64) if S6[i] >= 122.5 - 1e-9 and (i, b) in LP) for b in range(NB)]
SW3 = [b for b in range(1, NB) if ENV3[b] != ENV3[b - 1]]
PHI_IS_3 = all(abs(PHI[b] - LP[ENV3[b], b][0]) < 1e-9 for b in range(NB))
print("\nthe trade-off surface: the most payouts with savings of at least 122.5 a year and a reserve of at least b")
print(f"(always the best three-round plan, as more rounds pay less: {PHI_IS_3})")
runs = [0] + SW3 + [NB]
for lo, hi in zip(runs[:-1], runs[1:]):
    print(f"  reserves {lo:3d} to {hi - 1:3d}: rounds in years {years(SCHEDULES[ENV3[lo]])}")
B_SHOW = (0, 25, 50, 85, 86, 90, 100, 101, 105, 120, 150)
print("  reserve  " + "".join(f"{b:8d}" for b in B_SHOW))
print("  payouts  " + "".join(f"{PHI[b]:8.2f}" for b in B_SHOW))
SLOPES = [PHI[b - 1] - PHI[b] for b in range(1, NB) if b not in SW3]
DROPS = {b: PHI[b - 1] - PHI[b] - BETA for b in SW3}
print(f"where it is smooth, the surface falls by {min(SLOPES):.4f} to {max(SLOPES):.4f} a dollar, the multiplier; where the "
      f"rounds move it falls by more: " + "; ".join(f"at {b}, {PHI[b - 1] - PHI[b]:.2f} = {BETA:.2f} + {d:.2f}"
                                                     for b, d in DROPS.items()))

## Panel 4 — Three objectives and the dents (Theorem 12.2(iii); Worked Example 12.1, Step 4; Figure 12.2; Online Appendix Example 12.B.2: 992 plans, 740 efficient, 453 supported and 287 in dents, all with reserves from \$86m to \$149m; ideal point (384.17, 137.81, 150), nadir (22.06, 70, 0))

The three-objective frontier on the \$1 million grid of reserves (Online Appendix Remark 12.B.1). For each number of
rounds and each reserve only the best schedule can be efficient, since the savings depend only on the number of rounds:
992 plans. A plan is efficient when no other is at least as good for all three owners and better for one. An efficient
plan is **supported** when some weights, each at least 0.001 of their sum, with the objectives scaled by their ranges
over the efficient plans, make it the best weighted sum: one small linear program in the weights for each plan. The
cell also keeps a certificate of each answer that can be checked without a solver, and the workbook checks all 740: the
weights, for a supported plan; for an unsupported one, an average of efficient plans that is better in every objective,
which by Theorem 12.2(iii) rules out every choice of nonnegative weights, not all zero. The ideal point holds each
objective's best value; the nadir, each one's worst over the efficient plans.

In [ ]:
PTS = [(n, b, best_at(n, b)) for n in range(K + 1) for b in range(NB)]
PTS = [p for p in PTS if p[2] is not None]                   # (rounds, reserve, best schedule)
F3 = np.array([[LP[i, b][0], S6[i], b] for n, b, i in PTS])
EFF3 = ~dominated(F3)
E3 = [p for p, e in zip(PTS, EFF3) if e]                     # the efficient plans, by rounds, then reserve
FE = F3[EFF3]
IDEAL, NADIR = FE.max(axis=0), FE.min(axis=0)
SPAN = IDEAL - NADIR
GE = FE / SPAN                                                # the objectives scaled by their ranges
M = len(E3)
SUP = np.zeros(M, bool)
WEIGHTS, AVERAGES, MARGINS = {}, {}, {}
for k in range(M):
    # supported: weights w >= 0.001, summing to 1, with w . (G_j - G_k) <= 0 for every efficient plan j
    res = linprog(np.zeros(3), A_ub=GE - GE[k], b_ub=np.zeros(M), A_eq=np.ones((1, 3)), b_eq=[1.0],
                  bounds=[(1e-3, None)] * 3, method="highs")
    if res.status == 0:
        SUP[k], WEIGHTS[k] = True, res.x
        continue
    # unsupported: the average of efficient plans that beats it in every objective by the widest margin t
    res = linprog(np.r_[np.zeros(M), -1.0], A_ub=np.c_[-(GE - GE[k]).T, np.ones(3)], b_ub=np.zeros(3),
                  A_eq=np.r_[np.ones(M), 0.0][None, :], b_eq=[1.0], bounds=[(0, None)] * M + [(None, 1.0)],
                  method="highs-ds")
    lam = res.x[:M]
    AVERAGES[k] = [(int(j), float(lam[j])) for j in np.nonzero(lam > 1e-12)[0]]
    MARGINS[k] = float(res.x[-1])
CHECK_W = max(float(np.max((GE - GE[k]) @ w)) for k, w in WEIGHTS.items())
CHECK_A = min(float(np.min(sum(l * GE[j] for j, l in AVERAGES[k]) - GE[k])) for k in AVERAGES)
print(f"{len(PTS)} plans are the best for their number of rounds and reserve; {M} are efficient: {SUP.sum()} supported, "
      f"{M - SUP.sum()} in dents")
print(f"certificates: at its weights, no efficient plan beats a supported one (by at most {max(CHECK_W, 0.0):.0e}); each "
      f"plan in a dent is beaten in every objective by an average of at most {max(len(a) for a in AVERAGES.values())} "
      f"efficient plans, by at least {CHECK_A:.1e} of each range")
print("so no weights reach the dents even with zero components allowed: the answer is the same with weights merely "
      "nonnegative (Online Appendix Remark 12.B.1)")
print("\nrounds  efficient  supported  in dents  reserves of the plans in dents")
DENTS = {}
for n in range(K + 1):
    ks = [k for k in range(M) if E3[k][0] == n]
    if ks:
        DENTS[n] = [E3[k][1] for k in ks if not SUP[k]]
        print(f"{n:6d}  {len(ks):9d}  {sum(SUP[k] for k in ks):9d}  {len(DENTS[n]):8d}  "
              + (f"{min(DENTS[n])} to {max(DENTS[n])}" if DENTS[n] else "none"))
ALL_DENT_B = [E3[k][1] for k in range(M) if not SUP[k]]
print(f"every plan in a dent keeps a reserve from {min(ALL_DENT_B)} to {max(ALL_DENT_B)}")
KN = int(np.argmin(FE[:, 0]))
print(f"\nideal point: payouts {IDEAL[0]:.2f}, savings {IDEAL[1]:.4f}, reserve {IDEAL[2]:.0f}")
print(f"nadir point: payouts {NADIR[0]:.2f} (rounds in years {years(SCHEDULES[E3[KN][2]])} with a reserve of "
      f"{E3[KN][1]}), savings {NADIR[1]:g}, reserve {NADIR[2]:g}")
KX = next(k for k in range(M) if E3[k][:2] == (3, 92))
avg = sum(l * FE[j] for j, l in AVERAGES[KX])
print(f"\na plan in a dent: rounds in years {years(SCHEDULES[E3[KX][2]])} with a reserve of 92: payouts {FE[KX][0]:.2f}, "
      f"savings {FE[KX][1]:g}")
print("  is beaten in every objective by " + " + ".join(f"{l:.3f} x (years {years(SCHEDULES[E3[j][2]])}, reserve {E3[j][1]})"
                                                   for j, l in AVERAGES[KX])
      + f": payouts {avg[0]:.2f}, savings {avg[1]:.2f}, reserve {avg[2]:.2f}")

COLR = {1: "#9E9E9E", 2: "#1F4E79", 3: "#E65100", 4: "#2E7D32", 5: "#6A1B9A", 6: "#B71C1C"}
fig, ax = plt.subplots(figsize=(7.6, 3.6))
for n in range(1, K + 1):
    ks = sorted((k for k in range(M) if E3[k][0] == n), key=lambda k: E3[k][1])
    runs, cur = [], [ks[0]]
    for a, c in zip(ks[:-1], ks[1:]):              # runs of one schedule over consecutive reserves
        if E3[c][2] == E3[a][2] and E3[c][1] == E3[a][1] + 1:
            cur.append(c)
        else:
            runs.append(cur); cur = [c]
    runs.append(cur)
    for run in runs:
        bs, ps = np.array([E3[k][1] for k in run]), FE[run, 0]
        solid = [bool(SUP[run[a]] and SUP[run[a + 1]]) for a in range(len(run) - 1)]
        a = 0
        while a < len(solid):                      # one line for each stretch of the same style
            c = a
            while c + 1 < len(solid) and solid[c + 1] == solid[a]:
                c += 1
            ax.plot(bs[a:c + 2], ps[a:c + 2], color=COLR[n], lw=1.6, ls="-" if solid[a] else (0, (3, 2)))
            a = c + 1
    ax.plot([], [], color=COLR[n], lw=1.6, label=f"{n} round{'s' if n > 1 else ''}")
ax.plot([], [], color="#616161", lw=1.2, ls=(0, (3, 2)), label="dashed: no weights pick it")
for x in (85, 100):
    ax.axvline(x, color="#999999", lw=0.8, ls=":")
ax.text(84, 395, "rounds in years 1–3\nout of reach", fontsize=7.5, color="#616161", ha="right", va="top")
ax.text(101, 395, "rounds in years 1, 2\nout of reach", fontsize=7.5, color="#616161", ha="left", va="top")
ax.plot([92], [FE[KX][0]], marker="o", ms=6, color="black", ls="none", zorder=5)
ax.annotate("years 1, 2, 4, a \\$92m\nreserve: in a dent", xy=(92, FE[KX][0]), xytext=(106, 318), fontsize=8,
            arrowprops=dict(arrowstyle="-", lw=0.6, color="#333333"))
ax.set(xlim=(0, 150), ylim=(0, 400), xlabel="reserve on every cash commitment ($ million)",
       ylabel="payouts, present value ($ million)", title="Above $85m the rounds move later, into dents (seed 26212)")
ax.legend(frameon=False, fontsize=7.5, loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout(); plt.show()

## Panel 5 — Choose (Theorem 12.3; Proposition 12.1; Worked Example 12.1, Step 5; Table 12.3; Figure 12.3: equal weights choose rounds in years 1, 2 and 4 with a \$92m reserve, \$242.80m, a plan in a dent; the CFO weighted twice, two rounds with a \$96m reserve, \$296.90m; goals that can all be met: \$300.00m first, \$307.72m after a second stage)

AXIOM Lab 2.12, step 4. The board states aspirations instead of prices: the ideal point as the reference, each owner's
shortfall measured as a share of the range from the nadir to the ideal, and weights for their urgency. The augmented
Chebyshev program picks the efficient plan with the least largest weighted shortfall, plus $\rho = 10^{-4}$ times their
sum; every solution is efficient (Definition 12.2; Theorem 12.3(ii)). By Theorem 12.3(iii), with a reference point
above the ideal in every objective, every efficient plan is the choice for some positive weights, the plans in dents
included: the cell checks all 740. Each owner's own best optimizes that owner's objective first and the others after
(Proposition 12.1(iii)). Goal programming minimizes the weighted shortfalls below the board's goals, as shares of the
goals: payouts of at least \$300 million, savings of \$122.5 million a year and a \$25 million reserve. These goals can
all be met, so every plan that meets them is a solution, dominated ones included (Proposition 12.1(ii)). The book's
engine returns the first such schedule it meets, in binary order of the schedules, and pays just the goal, since the
shortfalls do not reward paying more; the second stage maximizes payouts among the plans that meet every goal.

In [ ]:
def chebyshev(w, rho=1e-4, ref=None):
    # the augmented Chebyshev choice: least largest weighted shortfall from the reference point, shortfalls as shares of
    # the ideal-nadir range, plus rho times their sum; the first on ties (by rounds, then reserve)
    ref = IDEAL if ref is None else ref
    d = np.asarray(w, float) * (ref - FE) / SPAN
    k = int(np.argmin(d.max(axis=1) + rho * d.sum(axis=1)))
    return k, d[k]

def plan(k):
    n, b, i = E3[k]
    return dict(k=k, n=n, b=b, i=i, z=SCHEDULES[i], P=float(FE[k][0]), S=float(FE[k][1]), p=LP[i, b][1],
                J=float(FE[k][0] + W_BOARD * FE[k][1]), supported=bool(SUP[k]))

def paid(p):
    return ", ".join(f"{x:g}" for x in np.round(p, 4))

CHOICE = {}
for key, w in (("equal", (1, 1, 1)), ("cfo2", (2, 1, 1))):
    k, d = chebyshev(w)
    CHOICE[key] = c = dict(plan(k), d=d, w=w)
    print(f"weights {w}: rounds in years {years(c['z'])}, reserve {c['b']}, payouts {c['P']:.2f} ({paid(c['p'])}), "
          f"savings {sv(c['S'])}; weighted shortfalls " + ", ".join(f"{x:.3f}" for x in d)
          + ("; supported" if c["supported"] else "; in a dent: no weights reach it"))
REF = IDEAL + 0.01 * SPAN
REACHED = 0
for k in range(M):
    w = SPAN / (REF - FE[k])                       # w_j = 1 / (r_j - f_j(x*)), in units of the ranges
    s = np.max(w * (REF - FE) / SPAN, axis=1)
    win = np.nonzero(s <= s.min() + 1e-12)[0]
    REACHED += bool(s[k] <= s.min() + 1e-12 and np.all(np.abs(FE[win] - FE[k]) < 1e-9))
print(f"Theorem 12.3(iii), the reference 1% of each range above the ideal: {REACHED} of {M} efficient plans are the only "
      f"choice for their own weights, the {M - SUP.sum()} in dents among them")

OWN = {"CFO": plan(max(range(M), key=lambda k: (round(FE[k][0], 9), FE[k][1], FE[k][2]))),
       "COO": plan(max(range(M), key=lambda k: (FE[k][1], round(FE[k][0], 9), FE[k][2]))),
       "treasury": plan(max(range(M), key=lambda k: (FE[k][2], round(FE[k][0], 9), FE[k][1])))}
print()
for nm, c in OWN.items():
    print(f"the {nm}'s own best: rounds in years {years(c['z'])}, reserve {c['b']}, payouts {c['P']:.2f} ({paid(c['p'])}), "
          f"savings {sv(c['S'])}; board value {c['J']:,.2f}")
BOARD3 = plan(max(range(M), key=lambda k: FE[k][0] + W_BOARD * FE[k][1]))
print(f"the board's weighted sum on the three-objective frontier: rounds in years {years(BOARD3['z'])}, reserve "
      f"{BOARD3['b']}, {BOARD3['J']:,.2f}")

GOALS = (300.0, 122.5, 25)
FIRST = None
for i in range(64):                                # the book's engine meets the schedules in binary order
    if (i, GOALS[2]) in LP:
        dev = max(0.0, GOALS[0] - LP[i, GOALS[2]][0]) / GOALS[0] + max(0.0, GOALS[1] - S6[i]) / GOALS[1]
        if FIRST is None or dev < FIRST[1] - 1e-12:
            FIRST = (i, dev)
G1 = FIRST[0]
G1_AVAIL = LP[G1, GOALS[2]][0]
G1_PAID = min(G1_AVAIL, GOALS[0])                  # the shortfalls do not reward paying more than the goal
MEET = [i for i in range(64) if (i, GOALS[2]) in LP and S6[i] >= GOALS[1] - 1e-9 and LP[i, GOALS[2]][0] >= GOALS[0] - 1e-9]
G2 = max(MEET, key=lambda i: (LP[i, GOALS[2]][0], -i))
G2_P = LP[G2, GOALS[2]][0]
G1_DOMINATED = dominates(np.array([G2_P, S6[G2], GOALS[2]]), np.array([G1_PAID, S6[G1], GOALS[2]]))
print(f"\ngoals {GOALS}: {len(MEET)} schedules meet them all: " + "; ".join(f"years {years(SCHEDULES[i])}, "
                                                                       f"{LP[i, GOALS[2]][0]:.2f}" for i in MEET))
print(f"the solver's first answer: rounds in years {years(SCHEDULES[G1])}, paying {G1_PAID:.2f} of the {G1_AVAIL:.2f} "
      f"available; dominated: {G1_DOMINATED}")
print(f"the second stage, the most payouts among the plans meeting every goal: rounds in years {years(SCHEDULES[G2])}, "
      f"{G2_P:.2f}; the first answer leaves {G2_P - G1_PAID:.2f} unpaid")

T123 = [("CFO's best", OWN["CFO"]["z"], OWN["CFO"]["b"], OWN["CFO"]["P"], OWN["CFO"]["S"]),
        ("COO's best", OWN["COO"]["z"], OWN["COO"]["b"], OWN["COO"]["P"], OWN["COO"]["S"]),
        ("Treasury's best", OWN["treasury"]["z"], OWN["treasury"]["b"], OWN["treasury"]["P"], OWN["treasury"]["S"]),
        ("Board value (Chapter 5)", SCHEDULES[BOARD_I], 0, P0[BOARD_I], S6[BOARD_I]),
        ("Chebyshev, equal weights", CHOICE["equal"]["z"], CHOICE["equal"]["b"], CHOICE["equal"]["P"], CHOICE["equal"]["S"]),
        ("Chebyshev, CFO twice", CHOICE["cfo2"]["z"], CHOICE["cfo2"]["b"], CHOICE["cfo2"]["P"], CHOICE["cfo2"]["S"]),
        ("Goals, first answer", SCHEDULES[G1], GOALS[2], G1_PAID, S6[G1]),
        ("Goals, second stage", SCHEDULES[G2], GOALS[2], G2_P, S6[G2])]
TABLE = [dict(name=nm, years=years(z), reserve=int(b), P=float(P), S=float(S), J=float(P + W_BOARD * S))
         for nm, z, b, P, S in T123]
print("\nTable 12.3: plans by owner and by method of choice (board value: payouts + 5.645 x savings)")
print(f"{'plan':26s} {'rounds (years)':15s} {'reserve':>7s} {'payouts':>8s} {'savings':>8s} {'board value':>12s}")
for r in TABLE:
    print(f"{r['name']:26s} {r['years']:15s} {r['reserve']:7d} {r['P']:8.2f} {r['S']:8.2f} {r['J']:12,.2f}")
J_RANGE = (min(r["J"] for r in TABLE), max(r["J"] for r in TABLE))
print(f"Figure 12.3's board-value axis runs from {J_RANGE[0]:,.2f} to {J_RANGE[1]:,.2f}")

## Panel 6 — Compute frontiers (Proposition 12.2; Table 12.2; Section 12.4: 9,664 payout programs and one support program for each of the 740 efficient plans; weights find only supported plans; an archive that keeps every undominated plan ends with the efficient set)

Section 12.4's methods on Meridian's instance. (i) **Bracketing**: with two objectives, bounds on savings box the
frontier, every efficient plan between two consecutive bounds lying between their two answers. (ii) **Hypervolume**:
the area a set of schedules dominates above a reference point below the nadir, here (0, 0); the sets of one, two and
three schedules that maximize it hold only efficient schedules, checked by trying every such set of the 64. (iii)
**Weights are not enough**: 10,000 random weights, each at least 0.001 of their sum, on the three scaled objectives,
drawn with the lab's seed 26212; every plan they return is supported, and none is in a dent. (iv) **Evolutionary
search**: in each generation each of the 992 plans is proposed with chance 0.02 (the same stream, seed 26212), and an
unbounded archive keeps every plan that no plan of the archive and the proposals dominates; by Proposition 12.2(iv) it
equals the efficient set after finitely many generations. An archive bounded below 740 plans never could, the
correction the book makes to the previous edition. These draws are the lab's own illustration; none of the book's
numbers depends on them.

In [ ]:
def phi2(eps):
    # the most payouts with savings of at least eps, no reserve
    return max(P0[i] for i in range(64) if S6[i] >= eps - 1e-9)

BOUNDS = [0.0, 60.0, 100.0, 110.0, 120.0, 125.0, 130.0, 134.0, 136.0, 137.8125]
BOXED = all(phi2(c) - 1e-9 <= P0[i] <= phi2(a) + 1e-9 for a, c in zip(BOUNDS[:-1], BOUNDS[1:]) for i in EFF2
            if a <= S6[i] <= c)
print("(i) bounds on savings and the most payouts they allow: " + ", ".join(f"{sv(e)}: {phi2(e):.2f}" for e in BOUNDS))
print(f"    every efficient plan between two consecutive bounds lies between their answers: {BOXED}")

def hv2(F, ref=(0.0, 0.0)):
    # the area that points (payouts, savings) dominate above the reference point
    F = np.asarray(F, float).reshape(-1, 2) - np.asarray(ref)
    F = F[(F[:, 0] > 0) & (F[:, 1] > 0)]
    if len(F) == 0:
        return 0.0
    F = F[np.argsort(-F[:, 1])]
    top = np.maximum.accumulate(F[:, 0])
    return float(np.sum((F[:, 1] - np.r_[F[1:, 1], 0.0]) * top))

HV_E = hv2(F2[EFF2])
HV_BEST, HV_OK = {}, True
for m_ in (1, 2, 3):
    best = max(itertools.combinations(range(64), m_), key=lambda s: hv2(F2[list(s)]))
    HV_BEST[m_] = (best, hv2(F2[list(best)]))
    HV_OK &= set(best) <= set(EFF2)
    print(f"(ii) the {m_} schedule{'s' if m_ > 1 else ''} with the largest hypervolume: "
          + "; ".join(f"years {years(SCHEDULES[i])}" for i in best) + f": {HV_BEST[m_][1]:,.2f}")
HV_ADD = all(hv2(F2[[e for e in EFF2 if e != x]]) < HV_E - 1e-9 for x in EFF2) and \
    all(abs(hv2(F2[EFF2 + [j]]) - HV_E) < 1e-9 for j in np.nonzero(DOM2)[0])
print(f"    the five efficient schedules: {HV_E:,.2f}; dropping any one lowers it, adding a dominated one does not raise "
      f"it: {HV_ADD}")

rng = np.random.default_rng(SEED)
WTS = 1e-3 + (1.0 - 3e-3) * rng.dirichlet(np.ones(3), 10_000)
G3 = F3 / SPAN
FOUND = sorted(set(int(np.argmax(G3 @ w)) for w in WTS))
POS = {p: k for k, p in enumerate(E3)}
FOUND_K = [POS.get(PTS[j]) for j in FOUND]
SWEEP_OK = all(k is not None and SUP[k] for k in FOUND_K)
print(f"(iii) {len(WTS):,} random weights (seed {SEED}) return {len(FOUND)} different plans, all efficient and supported: "
      f"{SWEEP_OK}; none of the {M - SUP.sum()} in dents")
print("     most supported plans lie inside stretches where the payouts fall at the reserve's price: a weighted sum is "
      "indifferent along a stretch only at one exact ratio of its weights, so random weights pick its ends")

def beats(Fa, Fb, tol=1e-9):
    # [a, b]: row a of Fa dominates row b of Fb
    ge = np.all(Fa[:, None, :] >= Fb[None, :, :] - tol, axis=2)
    gt = np.any(Fa[:, None, :] > Fb[None, :, :] + tol, axis=2)
    return ge & gt

EFFSET = set(np.nonzero(EFF3)[0].tolist())
archive, T_EVO, HIST = np.array([], int), None, {}
for t in range(1, 5001):
    prop = np.nonzero(rng.random(len(PTS)) < 0.02)[0]
    new = np.setdiff1d(prop, archive)
    pool = np.r_[archive, new]
    keep_old = ~beats(F3[new], F3[archive]).any(axis=0)
    keep_new = ~beats(F3[pool], F3[new]).any(axis=0)
    archive = np.r_[archive[keep_old], new[keep_new]]
    HIST[t] = (len(archive), len(EFFSET & set(archive.tolist())))
    if set(archive.tolist()) == EFFSET:
        T_EVO = t
        break
print(f"(iv) the unbounded archive (seed {SEED}), plans kept and how many of them efficient: "
      + "; ".join(f"generation {t}: {HIST[t][0]}, {HIST[t][1]}" for t in (1, 10, 50, 100, 200) if t in HIST))
print(f"     it equals the efficient set, all {len(EFFSET)} plans and nothing else, from generation {T_EVO}")
print(f"the book's computation: {N_PROGRAMS:,} payout programs ({len(LP):,} feasible) and {M} support programs")

## Panel 7 — Add uncertainty (Proposition 12.3; Section 12.5; Figure 10.3; Online Appendix Example 10.B.2: Chapter 9's cheapest committed packages cost \$12.35m at 4%, \$10.44m at 5% and \$8.82m at 6%, a point of appetite worth about \$1.6–1.9m; Chapter 11's records' family moves the 5% point to \$24.00m)

AXIOM Lab 2.12, step 5. Chapter 9's appetite, at most a 5 percent chance of a breach of the \$350 million floor in some
quarter, bounds one objective of a vector problem: the cost of a package and its chance of a breach. The least cost
$C(\alpha)$ at each appetite $\alpha$ is the trade-off surface; it never rises with the appetite, and its slope is the
appetite's price (Proposition 12.3(i)). Chapter 9's packages are entered here as the book prints them (Online Appendix
Example 10.B.2: a standby line and a committed delay for each appetite, on Chapter 9's 100,000 paths, seed 8); they are
declared values, not re-simulated. Declaring a family of laws never lowers $C(\alpha)$ (part (ii)): Chapter 11's
records' family keeps 5 percent at \$24.00 million (Worked Example 11.1). Part (iii): with conditional value at risk
the frontier is convex and weighted sums reach all of it, but the chance itself can have dents, as the small example of
Online Appendix 12.B shows.

In [ ]:
# Chapter 9's cheapest committed packages as Online Appendix Example 10.B.2 prints them (declared, not re-simulated):
# appetite (%), standby line ($m), committed delay (quarters), cost ($m)
CH9 = [(1.0, 190, 2.174, 23.09), (2.0, 170, 1.595, 17.86), (3.0, 180, 1.161, 14.68), (4.0, 165, 0.925, 12.35),
       (5.0, 170, 0.668, 10.44), (6.0, 160, 0.502, 8.82), (8.0, 160, 0.182, 6.25), (10.0, 135, 0.015, 4.17),
       (12.5, 100, 0.0, 3.00), (15.0, 75, 0.0, 2.25)]
FEE, VQ = 0.03, 8.0              # Chapter 9's levers: line fees of 3% over the program; $8m a quarter of delay
FAMILY_5 = 24.00                 # Chapter 11's records' family keeps 5% at $24.00m (Worked Example 11.1)
A9 = np.array([r[0] for r in CH9])
C9 = np.array([r[3] for r in CH9])
LEVERS_GAP = max(abs(FEE * l + VQ * d - c) for _, l, d, c in CH9)
print(f"each declared cost is 3% of the line plus $8m a quarter of delay, within {LEVERS_GAP:.3f} (delays printed to three decimals)")
SL = -np.diff(C9) / np.diff(A9)                  # $m of cost per point of appetite, between neighbours
print("appetite (%)    " + "".join(f"{a:7g}" for a in A9))
print("least cost      " + "".join(f"{c:7.2f}" for c in C9))
print("price a point      " + "".join(f"{s:7.2f}" for s in SL))
P45, P56 = C9[3] - C9[4], C9[4] - C9[5]
print(f"near the board's 5%: a point of appetite is worth {P56:.2f} (5% to 6%) to {P45:.2f} (4% to 5%): about "
      f"{P56:.1f} to {P45:.1f} million")
MONO, FALLING = bool(np.all(np.diff(C9) < 0)), bool(np.all(np.diff(SL) < 0))
print(f"Proposition 12.3(i): the least cost never rises with the appetite: {MONO}; on these ten points its price falls "
      f"as the appetite grows: {FALLING}")
print(f"Proposition 12.3(ii): the records' family keeps 5% at {FAMILY_5:.2f}, against {C9[4]:.2f} under one law: the "
      f"family moves the point up by {FAMILY_5 - C9[4]:.2f}, Chapter 11's price of ambiguity")

def chance(z):
    # Online Appendix 12.B's example: a line of size z in [0, 2] costs z; the need is 1 or 2 with chances 0.3 and 0.7
    return 1.0 if z < 1 else (0.7 if z < 2 else 0.0)

AVG = (0.55 * 0.0 + 0.45 * 2.0, 0.55 * chance(0.0) + 0.45 * chance(2.0))
print(f"Proposition 12.3(iii): the line z = 1 (cost 1, chance {chance(1.0):g}) is efficient, yet 0.55 x (0, 1) + "
      f"0.45 x (2, 0) = ({AVG[0]:.1f}, {AVG[1]:.2f}) is better in both: a dent, which no weighted sum reaches")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.plot(A9, C9, "o-", color="#1F4E79", lw=1.8, ms=4, label="Chapter 9's frontier: the cheapest committed package")
ax.plot([5.0], [FAMILY_5], marker="s", ms=7, color="#E65100", ls="none", label="Chapter 11's records' family, at 5%")
ax.annotate("", xy=(5.0, FAMILY_5 - 0.6), xytext=(5.0, C9[4] + 0.6),
            arrowprops=dict(arrowstyle="->", color="#E65100", lw=0.9))
ax.text(5.3, (FAMILY_5 + C9[4]) / 2, f"family: +\\${FAMILY_5 - C9[4]:.2f}m", fontsize=8, color="#E65100", va="center")
ax.text(4.4, 7.6, f"4% to 5%:\n\\${P45:.2f}m a point", fontsize=8, color="#1F4E79", ha="right", va="center")
ax.text(6.2, C9[5] + 1.1, f"5% to 6%: \\${P56:.2f}m a point", fontsize=8, color="#1F4E79", ha="left", va="center")
ax.axvline(5.0, color="#999999", lw=0.8, ls=":")
ax.set(xlim=(0, 16), ylim=(0, 26), xlabel="appetite: chance of a breach in some quarter (%)",
       ylabel="cost of the package ($ million)", title="Near 5%, a point of appetite is worth \\$1.6m to \\$1.9m (seed 26212)")
ax.legend(frameon=False, fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()

## Validation — the book's numbers, and agreement with `DCT_V2_Ch12_Lab.xlsx`

Each checkpoint is a number printed in Chapter 12 (Sections 12.1–12.5, Worked Example 12.1, Table 12.3, Figures
12.1–12.3, the LOS Guidance) or in Online Appendix Example 12.B.2, with a tolerance of half a unit in its last printed
digit; a few check what a theorem or proposition promises (1 = holds). The five-round plan's savings, \$135.625
million a year, print as 135.63 in Step 2 and pass at its last printed digit. Chapter 9's frontier is the book's
declared values (Online Appendix Example 10.B.2), so its checkpoints check the prices the chapter reads from it. The
workbook recomputes the checkpoints marked *workbook* with spreadsheet formulas and shows PASS against the same engine
values on its `Checks` sheet.

In [ ]:
def bk(s):
    # a number as the book prints it: its value and half a unit in its last printed digit
    t = s.replace(",", "")
    dp = len(t.split(".")[1]) if "." in t else 0
    return float(t), 0.5 * 10.0 ** -dp + 1e-9

def c_(name, got, printed):
    v, tol = bk(printed)
    return (name, float(got), v, tol)

def flag(name, ok):
    return (name, float(bool(ok)), 1.0, 0.0)

def count(name, got, n):
    return (name, float(got), float(n), 0.0)

WORD = {0: "no rounds", 1: "one round", 2: "two rounds", 3: "three rounds", 4: "four rounds", 5: "five rounds", 6: "six rounds"}
OA_BEST = {1: ("370.67", "70", ("220", "70", "70", "70", "20", "0")), 2: ("384.17", "105", ("100", "105", "105", "105", "90", "0")),
           3: ("330.45", "122.5", ("85", "0", "122.5", "122.5", "122.5", "2.5")),
           4: ("260.88", "131.25", ("85", "0", "122.5", "122.5", "2.5", "11.25")),
           5: ("186.82", "135.625", ("85", "0", "122.5", "2.5", "11.25", "15.625")),
           6: ("106.59", "137.8125", ("85", "0", "2.5", "11.25", "15.625", "17.8125"))}
checks = [
    # the declarations and the payout programs
    count("payout programs, 64 schedules x 151 reserves (workbook)", N_PROGRAMS, 9664),
    count("payout programs that keep every commitment (workbook)", len(LP), 7824),
    flag("war chest = linear program: the same feasibility in all 9,664", WC_SAME),
    ("war chest = linear program: largest difference", WC_GAP, 0.0, 1e-9),
    flag("savings depend only on the rounds: 140 - 140/2^n (workbook)", np.allclose(S6, 140 - 140 / 2.0 ** NR)),
    c_("Chapter 5's price of savings, Section 12.2 (workbook)", W_BOARD, "5.645"),
    c_("Chapter 5's price of savings, Online Appendix (workbook)", W_BOARD, "5.6447"),
    # Step 2; Figure 12.1(a) (Panel 1)
    count("schedules keeping every commitment, no reserve (workbook)", N_FEAS0, 64),
    count("dominated schedules (workbook)", DOM2.sum(), 59),
    count("efficient schedules (workbook)", len(EFF2), 5),
    flag("the efficient plans are the best with two to six rounds", [NR[i] for i in EFF2] == [2, 3, 4, 5, 6]
         and all(BEST_N[n] in EFF2 for n in range(2, 7))),
    c_("no rounds: payouts (workbook)", P0[BEST_N[0]], "136.36"),
    c_("no rounds: payout in year 1 (workbook)", LP[BEST_N[0], 0][1][0], "150"),
]
for n, (pv, sv, pays) in OA_BEST.items():
    i = BEST_N[n]
    checks += [c_(f"{WORD[n]}: payouts (workbook)", P0[i], pv), c_(f"{WORD[n]}: savings (workbook)", S6[i], sv)]
    checks += [c_(f"{WORD[n]}: payout in year {k + 1} (workbook)", LP[i, 0][1][k], s) for k, s in enumerate(pays)]
checks += [c_(f"three rounds: cash at the end of year {k + 1} (workbook)", LP[I3, 0][2][k], s)
           for k, s in enumerate(("485", "470", "350", "350", "350", "470"))]
checks += [
    c_("five rounds: savings as Step 2 prints them", S6[BEST_N[5]], "135.63"),
    c_("six rounds: savings as Step 2 prints them", S6[BEST_N[6]], "137.81"),
    c_("three rounds: paid in all, Worked Example 12.1 (workbook)", LP[I3, 0][1].sum(), "455"),
    flag("one round is dominated by two (workbook)", dominates(F2[BEST_N[2]], F2[BEST_N[1]])),
    flag("Theorem 12.1(i): every dominated schedule is dominated by an efficient one", COVERED),
    c_("ideal point, two objectives: payouts (workbook)", IDEAL2[0], "384.17"),
    c_("ideal point, two objectives: savings (workbook)", IDEAL2[1], "137.81"),
    flag("Theorem 12.1(iii): the ideal point is out of reach (workbook)", not IDEAL2_HIT),
    c_("bound: savings of at least 122.5, payouts (workbook)", LP[EPS_A, 0][0], "330.45"),
    flag("bound: savings of at least 122.5 picks years 1-3, the only answer", SCHEDULES[EPS_A] == (1, 1, 1, 0, 0, 0) and EPS_UNIQUE),
    c_("bound: savings 122.5 and a reserve of 50, payouts (workbook)", LP[EPS_B, 50][0], "285.00"),
]
for r, (dp, ds, rate) in zip(RATES, (("53.72", "17.5", "3.07"), ("69.57", "8.75", "7.95"), ("74.06", "4.375", "16.93"),
                                     ("80.23", "2.1875", "36.68"))):
    nm = f"{r['frm']} to {r['to']} rounds"
    checks += [c_(f"{nm}: payouts given up (workbook)", r["dP"], dp), c_(f"{nm}: savings gained (workbook)", r["dS"], ds),
               c_(f"{nm}: price of savings (workbook)", r["rate"], rate)]
checks += [c_(f"board value, {WORD[n]} (workbook)", BOARD_V[n], s)
           for n, s in ((2, "976.87"), (3, "1,021.93"), (4, "1,001.75"), (5, "952.39"), (6, "884.51"))]
checks += [
    count("the board's weighted sum: rounds (workbook)", NR[BOARD_I], 3),
    flag("the board's weighted sum: years 1-3", SCHEDULES[BOARD_I] == (1, 1, 1, 0, 0, 0)),
    flag("the prices rise along the two-objective frontier (workbook)", INCREASING),
    # Step 4; Figure 12.2 (Panel 3)
    c_("the reserve's price: smallest multiplier (workbook)", PRICES.min(), "0.9091"),
    c_("the reserve's price: largest multiplier (workbook)", PRICES.max(), "0.9091"),
    c_("the reserve's price, Step 4 (workbook)", BETA, "0.909"),
    c_("cash before a year-2 round, after a year-1 round (workbook)", CASH2, "570"),
    c_("rounds in years 1 and 2 need a reserve of at most (workbook)", CASH2 - PRE, "100"),
    c_("cash before a year-3 round, after rounds in years 1 and 2 (workbook)", CASH3, "555"),
    c_("rounds in years 1-3 need a reserve of at most (workbook)", CASH3 - PRE, "85"),
    count("no plan keeps a reserve above (workbook)", BMAX.max(), 150),
    flag("the trade-off surface for savings of 122.5 is the three-round plans'", PHI_IS_3),
    count("three rounds: first switch (workbook)", SW3[0], 86),
    count("three rounds: second switch (workbook)", SW3[1], 101),
    count("three rounds: switches", len(SW3), 2),
    flag("three rounds: years 1-3 up to 85", ENV3[0] == ENV3[85] == SCHEDULES.index((1, 1, 1, 0, 0, 0))),
    flag("three rounds: years 1, 2, 4 from 86 to 100", ENV3[86] == ENV3[100] == SCHEDULES.index((1, 1, 0, 1, 0, 0))),
    flag("three rounds: years 1, 3, 4 from 101 to 150", ENV3[101] == ENV3[150] == SCHEDULES.index((1, 0, 1, 1, 0, 0))),
]
checks += [c_(f"three rounds at a reserve of {b} (workbook)", PHI[b], s) for b, s in
           ((25, "307.72"), (50, "285.00"), (85, "253.18"), (86, "248.25"), (90, "244.61"), (100, "235.52"),
            (101, "210.23"), (120, "192.96"), (150, "165.69"))]
checks += [
    c_("drop beyond the price at 86 (workbook)", DROPS[86], "4.02"),
    c_("drop beyond the price at 101 (workbook)", DROPS[101], "24.38"),
    ("Theorem 12.2(iv): the surface falls at the multiplier where smooth", max(abs(s - BETA) for s in SLOPES), 0.0, 1e-9),
    # three objectives (Panel 4)
    count("plans best for their rounds and reserve (workbook)", len(PTS), 992),
    count("efficient plans, three objectives (workbook)", M, 740),
    count("supported plans (workbook)", SUP.sum(), 453),
    count("plans in dents (workbook)", M - SUP.sum(), 287),
]
for n, (lo, hi, k) in {1: (101, 149, 49), 2: (101, 149, 49), 3: (86, 149, 63), 4: (86, 149, 63), 5: (86, 149, 63)}.items():
    checks += [count(f"dents, {WORD[n]}: plans (workbook)", len(DENTS[n]), k),
               count(f"dents, {WORD[n]}: smallest reserve (workbook)", min(DENTS[n]), lo),
               count(f"dents, {WORD[n]}: largest reserve (workbook)", max(DENTS[n]), hi)]
checks += [
    count("dents, six rounds: none (workbook)", len(DENTS[6]), 0),
    count("dents: smallest reserve, Step 4 (workbook)", min(ALL_DENT_B), 86),
    count("dents: largest reserve, Step 4 (workbook)", max(ALL_DENT_B), 149),
    c_("ideal point: payouts (workbook)", IDEAL[0], "384.17"),
    c_("ideal point: savings (workbook)", IDEAL[1], "137.81"),
    c_("ideal point: reserve (workbook)", IDEAL[2], "150"),
    c_("nadir point: payouts (workbook)", NADIR[0], "22.06"),
    c_("nadir point: savings (workbook)", NADIR[1], "70"),
    c_("nadir point: reserve (workbook)", NADIR[2], "0"),
    flag("nadir payouts: five rounds in years 1, 3-6 with a reserve of 150", E3[KN][1] == 150 and SCHEDULES[E3[KN][2]] == (1, 0, 1, 1, 1, 1)),
    flag("certificates: every supported plan is the best at its weights (workbook)", CHECK_W <= 1e-9),
    flag("certificates: every plan in a dent is beaten by an average (workbook)", CHECK_A > 0),
    count("Section 12.4: support programs, one for each efficient plan", M, 740),
]
CE, C2 = CHOICE["equal"], CHOICE["cfo2"]
checks += [
    # Step 5; Table 12.3; Figure 12.3 (Panel 5)
    flag("equal weights: rounds in years 1, 2, 4 (workbook)", CE["z"] == (1, 1, 0, 1, 0, 0)),
    count("equal weights: reserve (workbook)", CE["b"], 92),
    c_("equal weights: payouts (workbook)", CE["P"], "242.80"),
    c_("equal weights: savings (workbook)", CE["S"], "122.5"),
] + [c_(f"equal weights: payout in year {k + 1} (workbook)", CE["p"][k], s)
     for k, s in enumerate(("8", "90", "0", "122.5", "122.5", "2.5"))] + [
    c_("equal weights: shortfall, payouts (workbook)", CE["d"][0], "0.390"),
    c_("equal weights: shortfall, savings (workbook)", CE["d"][1], "0.226"),
    c_("equal weights: shortfall, reserve (workbook)", CE["d"][2], "0.387"),
    flag("equal weights: in a dent (workbook)", not CE["supported"]),
    c_("equal weights: board value (workbook)", CE["J"], "934.28"),
    flag("CFO twice: rounds in years 1, 2 (workbook)", C2["z"] == (1, 1, 0, 0, 0, 0)),
    count("CFO twice: reserve (workbook)", C2["b"], 96),
    c_("CFO twice: payouts (workbook)", C2["P"], "296.90"),
    c_("CFO twice: savings (workbook)", C2["S"], "105"),
] + [c_(f"CFO twice: payout in year {k + 1} (workbook)", C2["p"][k], s)
     for k, s in enumerate(("4", "105", "105", "105", "90", "0"))] + [
    c_("CFO twice: weighted shortfall, payouts (workbook)", C2["d"][0], "0.482"),
    c_("CFO twice: weighted shortfall, savings (workbook)", C2["d"][1], "0.484"),
    c_("CFO twice: weighted shortfall, reserve (workbook)", C2["d"][2], "0.360"),
    flag("CFO twice: supported (workbook)", C2["supported"]),
    c_("CFO twice: board value (workbook)", C2["J"], "889.60"),
    count("Theorem 12.3(iii): efficient plans each the only choice for its weights (workbook)", REACHED, 740),
    flag("CFO's best: years 1, 2 (workbook)", OWN["CFO"]["z"] == (1, 1, 0, 0, 0, 0)),
    flag("COO's best: all six (workbook)", OWN["COO"]["z"] == (1, 1, 1, 1, 1, 1)),
    flag("treasury's best: year 1 (workbook)", OWN["treasury"]["z"] == (1, 0, 0, 0, 0, 0)),
] + [c_(f"treasury's best: payout in year {k + 1} (workbook)", OWN["treasury"]["p"][k], s)
     for k, s in enumerate(("70", "70", "70", "70", "20", "0"))] + [
    flag("the board's weighted sum, three objectives: years 1-3, no reserve", BOARD3["z"] == (1, 1, 1, 0, 0, 0) and BOARD3["b"] == 0),
    c_("the board's weighted sum, three objectives: value (workbook)", BOARD3["J"], "1,021.93"),
    flag("goals: first answer, years 1, 2, 5 (workbook)", SCHEDULES[G1] == (1, 1, 0, 0, 1, 0)),
    c_("goals: first answer pays (workbook)", G1_PAID, "300.00"),
    c_("goals: first answer, available (workbook)", G1_AVAIL, "300.05"),
    flag("goals: the first answer is dominated (workbook)", G1_DOMINATED),
    flag("goals: second stage, years 1-3 (workbook)", SCHEDULES[G2] == (1, 1, 1, 0, 0, 0)),
    c_("goals: second stage pays (workbook)", G2_P, "307.72"),
    c_("goals: left unpaid, LOS Guidance 12.4 (workbook)", G2_P - G1_PAID, "7.72"),
]
T123_BOOK = {"CFO's best": ("1, 2", "0", "384.17", "105.00", "976.87"), "COO's best": ("1–6", "0", "106.59", "137.81", "884.51"),
             "Treasury's best": ("1", "150", "234.31", "70.00", "629.44"),
             "Board value (Chapter 5)": ("1–3", "0", "330.45", "122.50", "1,021.93"),
             "Chebyshev, equal weights": ("1, 2, 4", "92", "242.80", "122.50", "934.28"),
             "Chebyshev, CFO twice": ("1, 2", "96", "296.90", "105.00", "889.60"),
             "Goals, first answer": ("1, 2, 5", "25", "300.00", "122.50", "991.48"),
             "Goals, second stage": ("1–3", "25", "307.72", "122.50", "999.21")}
for r in TABLE:
    yb, rb, pb, sb, jb = T123_BOOK[r["name"]]
    checks += [flag(f"Table 12.3, {r['name']}: rounds", r["years"] == yb),
               c_(f"Table 12.3, {r['name']}: reserve (workbook)", r["reserve"], rb),
               c_(f"Table 12.3, {r['name']}: payouts (workbook)", r["P"], pb),
               c_(f"Table 12.3, {r['name']}: savings (workbook)", r["S"], sb),
               c_(f"Table 12.3, {r['name']}: board value (workbook)", r["J"], jb)]
checks += [
    c_("Figure 12.3: board value from (workbook)", J_RANGE[0], "629.44"),
    c_("Figure 12.3: board value to (workbook)", J_RANGE[1], "1,021.93"),
    # Section 12.4; Proposition 12.2 (Panel 6)
    flag("Proposition 12.2(i): bounds box the frontier", BOXED),
    flag("Proposition 12.2(ii): sets of 1-3 schedules with the most hypervolume are efficient", HV_OK),
    flag("Proposition 12.2(ii): an undominated schedule adds hypervolume, a dominated one none", HV_ADD),
    flag("Proposition 12.2(iii): random positive weights return only supported plans", SWEEP_OK),
    flag("Proposition 12.2(iv): the unbounded archive ends with the efficient set", T_EVO is not None),
    # Section 12.5; Proposition 12.3 (Panel 7)
    c_("Chapter 9's frontier: a point of appetite, 4% to 5% (workbook)", P45, "1.9"),
    c_("Chapter 9's frontier: a point of appetite, 5% to 6% (workbook)", P56, "1.6"),
    flag("Proposition 12.3(i): the least cost never rises with the appetite (workbook)", MONO),
    flag("declared costs: 3% of the line plus $8m a quarter, within the delays' rounding (workbook)", LEVERS_GAP <= 8 * 0.0005 + 0.005),
    c_("Proposition 12.3(ii): the family above one law, Worked Example 11.1 (workbook)", FAMILY_5 - C9[4], "13.56"),
    c_("Proposition 12.3(iii): the average's cost (workbook)", AVG[0], "0.9"),
    c_("Proposition 12.3(iii): the average's chance (workbook)", AVG[1], "0.55"),
]
ok = 0
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol else "FAIL"
    ok += status == "PASS"
    print(f"{name:84s} {got:11.4f}   book {want:10.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 13, *Machine Learning for Enterprise Transformation*, turns from models the board declares to models
learned from the enterprise's own data. Chapters 1–12 optimized declared models: dynamics written down, laws and
families of laws declared, and, in this chapter, objectives stated by their owners and chosen between on a frontier.
AXIOM-12 runs the same instance interactively (Lab 2.12); this notebook and the workbook are its reproducible record.